# 随机森林手势分类：训练结果分析

## 概述

本 Notebook 展示随机森林模型在 TENG 手势分类任务上的完整训练和评估结果，包括：

1. **数据划分方案**：按 (环境, 手势) 分层 80/20 划分，确保每个环境在训练集和测试集中都有样本
2. **整体分类性能**：混淆矩阵、precision/recall/F1
3. **各环境单独评估**：对比模型在 base、wind_noise、uv_radiation 三种环境下的表现
4. **特征重要度分析**：哪些特征对分类贡献最大
5. **误分类分析**：哪些手势容易混淆、为什么混淆
6. **模型参数敏感性**：树数量对性能的影响

### 模型配置

| 参数 | 值 |
|------|-----|
| 模型 | RandomForestClassifier (scikit-learn) |
| 决策树数量 | 200 |
| 最大特征数 | sqrt(9) = 3 |
| 类别权重 | balanced（自动补偿类别不平衡） |
| 输入特征 | 9D: [MAV x3, WL x3, Ratio x3] |
| 输出 | 10 类手势 |

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    ConfusionMatrixDisplay, f1_score
)
from sklearn.ensemble import RandomForestClassifier

from src.preprocess.io import GESTURE_NAMES
from src.preprocess.features import FEATURE_NAMES
from src.dataset import load_and_split

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 9

FEATURES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'features' / 'all_features.csv'
MODEL_PATH = PROJECT_ROOT / 'checkpoints' / 'random_forest.pkl'
SCALER_PATH = PROJECT_ROOT / 'checkpoints' / 'scaler.pkl'

---
## 1. 数据划分

### 划分策略

为了同时满足以下要求：
- **训练集尽量大**（最大化模型学习的数据量）
- **每个环境在训练集和测试集中都有样本**（可评估各环境的分类性能）

采用 **按 (环境, 手势) 组合分层抽样** 的方式：对每个 (环境, 手势) 组合，
随机抽取约 20% 作为测试集，剩余 80% 作为训练集。
分层抽样保证了每种手势在每个环境中都有训练和测试样本，避免分布偏移。

In [ ]:
X_train, X_test, y_train, y_test, scaler, df = load_and_split(FEATURES_PATH)

train_df = df[df['split'] == 'train']
test_df = df[df['split'] == 'test']

In [ ]:
# 划分详情：各环境 x 各手势的 train/test 数量
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
envs = ['base', 'wind_noise', 'uv_radiation']
env_titles = ['Base (Normal)', 'Wind Noise', 'UV Radiation']

for ax, env, title in zip(axes, envs, env_titles):
    env_train = train_df[train_df['env'] == env]['gesture_name'].value_counts().sort_index()
    env_test = test_df[test_df['env'] == env]['gesture_name'].value_counts().sort_index()
    
    # Align indices
    all_gestures = sorted(df['gesture_name'].unique())
    train_counts = [env_train.get(g, 0) for g in all_gestures]
    test_counts = [env_test.get(g, 0) for g in all_gestures]
    
    x = np.arange(len(all_gestures))
    w = 0.35
    ax.bar(x - w/2, train_counts, w, label='Train', color='#64B5F6', edgecolor='white')
    ax.bar(x + w/2, test_counts, w, label='Test', color='#FF8A65', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(all_gestures, rotation=45, fontsize=7)
    ax.set_title(f'{title}\n(train={sum(train_counts)}, test={sum(test_counts)})', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

axes[0].set_ylabel('Number of Samples')
fig.suptitle('Train/Test Split by Environment x Gesture (Stratified 80/20)', fontsize=12)
plt.tight_layout()
plt.show()

### 如何读这张图

三个子图分别对应三种环境，每个子图中：
- **蓝色柱**：训练集样本数
- **橙色柱**：测试集样本数

关键观察：
- 每种手势在每个环境中都有训练和测试样本（没有空缺）
- 训练集约为测试集的 4 倍（80/20 比例）
- base 环境样本最多（合并了两批次数据），uv_radiation 最少（每文件仅约 5 次动作）

---
## 2. 整体分类性能

### 加载模型并预测

In [ ]:
model = joblib.load(MODEL_PATH)
y_pred = model.predict(X_test)

overall_acc = accuracy_score(y_test, y_pred)
overall_f1 = f1_score(y_test, y_pred, average='macro')

print(f'Overall Test Accuracy: {overall_acc:.3f}')
print(f'Overall Macro F1:      {overall_f1:.3f}')
print(f'Train Accuracy:        {model.score(X_train, y_train):.3f}')

In [ ]:
# Classification Report
present_labels = sorted(set(y_test) | set(y_pred))
present_names = [GESTURE_NAMES[i] for i in present_labels]

report = classification_report(y_test, y_pred,
                               labels=present_labels,
                               target_names=present_names,
                               output_dict=True)
report_df = pd.DataFrame(report).T

# 只保留各手势行
gesture_report = report_df.loc[present_names].copy()
gesture_report['support'] = gesture_report['support'].astype(int)

print('Classification Report:')
print(classification_report(y_test, y_pred,
                            labels=present_labels,
                            target_names=present_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=present_labels)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=present_names, yticklabels=present_names,
            ax=ax, linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.set_title(f'Confusion Matrix (Test Accuracy: {overall_acc:.3f}, Macro F1: {overall_f1:.3f})', fontsize=12)
plt.tight_layout()
plt.show()

### 如何读混淆矩阵

- **行**：真实标签（ground truth）
- **列**：模型预测标签
- **对角线**：正确分类的样本数（颜色越深越好）
- **非对角线**：误分类的样本数（理想情况下全为 0）

例如，如果第 "4" 行、"ok" 列的值为 3，表示有 3 个真实标签为 "4" 的样本被错误预测为 "ok"。
对角线数值之和除以总样本数即为整体准确率。

In [ ]:
# Per-gesture F1 bar chart
fig, ax = plt.subplots(figsize=(10, 5))
f1_scores = gesture_report['f1-score'].values
colors = ['#66BB6A' if f >= 0.7 else '#FFA726' if f >= 0.5 else '#EF5350' for f in f1_scores]

bars = ax.bar(present_names, f1_scores, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(0.7, color='green', linestyle='--', linewidth=0.8, alpha=0.5, label='Good (0.7)')
ax.axhline(0.5, color='orange', linestyle='--', linewidth=0.8, alpha=0.5, label='Fair (0.5)')
ax.set_ylabel('F1 Score')
ax.set_xlabel('Gesture')
ax.set_title('Per-gesture F1 Score')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

for bar, f1, sup in zip(bars, f1_scores, gesture_report['support'].values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{f1:.2f}\n(n={sup})', ha='center', fontsize=7)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 如何读这张图

每根柱子代表一种手势的 F1 分数（precision 和 recall 的调和平均）：
- **绿色 (>=0.7)**：分类效果好
- **橙色 (0.5-0.7)**：分类效果一般，有改进空间
- **红色 (<0.5)**：分类效果差，需要重点关注

柱子上方标注了 F1 值和测试样本数 (n=)。
样本数少的手势（如 uv_radiation 环境）F1 波动较大，需谨慎解读。

---
## 3. 各环境单独评估

模型需要在不同噪声条件下都能有效工作。
分别查看三种环境的分类准确率和混淆矩阵，了解噪声对分类性能的影响。

In [ ]:
test_df_with_pred = test_df.copy()
test_df_with_pred['pred'] = y_pred
test_df_with_pred['correct'] = test_df_with_pred['label'] == test_df_with_pred['pred']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, env, title in zip(axes, envs, env_titles):
    env_df = test_df_with_pred[test_df_with_pred['env'] == env]
    if len(env_df) == 0:
        continue
    
    env_labels = env_df['label'].values
    env_preds = env_df['pred'].values
    env_acc = accuracy_score(env_labels, env_preds)
    
    env_present = sorted(set(env_labels) | set(env_preds))
    env_names = [GESTURE_NAMES[i] for i in env_present]
    cm_env = confusion_matrix(env_labels, env_preds, labels=env_present)
    
    sns.heatmap(cm_env, annot=True, fmt='d', cmap='Blues',
                xticklabels=env_names, yticklabels=env_names,
                ax=ax, linewidths=0.5, linecolor='white')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{title}\nAccuracy: {env_acc:.3f} (n={len(env_df)})', fontsize=10)
    ax.tick_params(axis='both', labelsize=7)

fig.suptitle('Confusion Matrix by Environment', fontsize=12)
plt.tight_layout()
plt.show()

print('Per-environment accuracy:')
for env, title in zip(envs, env_titles):
    env_df = test_df_with_pred[test_df_with_pred['env'] == env]
    acc = accuracy_score(env_df['label'], env_df['pred'])
    print(f'  {title:20s}: {acc:.3f} ({len(env_df)} samples)')

### 如何读这张图

三个混淆矩阵分别展示模型在三种环境下的表现：

- **Base（正常环境）**：基准性能，反映模型对干净信号的分类能力
- **Wind Noise（风噪）**：如果性能下降，说明风噪影响了特征的区分度
- **UV Radiation（紫外辐照）**：如果性能下降，说明辐照改变了传感器特性

对比三个矩阵的对角线占比，可以直观感受不同噪声条件的影响程度。
注意 uv_radiation 的测试样本最少（约 12 个），单个误分类就会显著影响准确率。

---
## 4. 特征重要度分析

随机森林的一个重要优势是可以输出 **特征重要度**（Gini importance），
反映每个特征在决策树分裂时对信息增益的贡献。
这有助于理解模型的决策依据，也为后续特征工程提供方向。

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)  # ascending for horizontal bar

fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(FEATURE_NAMES)))
bars = ax.barh(range(len(FEATURE_NAMES)),
               importances[indices],
               color=colors)
ax.set_yticks(range(len(FEATURE_NAMES)))
ax.set_yticklabels([FEATURE_NAMES[i] for i in indices])
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Random Forest Feature Importance')
ax.grid(True, alpha=0.3, axis='x')

# 标注数值
for bar, imp in zip(bars, importances[indices]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{imp:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print('Feature importance ranking:')
for rank, idx in enumerate(indices[::-1], 1):
    print(f'  {rank}. {FEATURE_NAMES[idx]:12s} = {importances[idx]:.4f}')

### 如何读这张图

- **横轴**：特征重要度（Gini importance），值越大表示该特征对分类贡献越大
- **纵轴**：9 个特征，按重要度从低到高排列

### 特征解读

- **Ratio 特征**（各通道能量占比）：反映手势在空间上的能量分布。
  不同手势在不同通道上的响应强度不同，因此 Ratio 特征对区分手势很有效。
- **WL 特征**（波形长度）：反映信号的变化速率和复杂度。
  某些手势（如 wave）的动作更复杂，WL 值更高。
- **MAV 特征**（平均绝对值）：反映整体信号强度。
  不同手势的力度差异通过 MAV 体现。

如果某类特征（如 Ratio）整体重要度较高，说明空间分布是区分手势的关键信息。

---
## 5. 误分类分析

分析模型最容易混淆的手势对，理解误分类的原因。

In [ ]:
# 找出最常见的误分类对
errors = test_df_with_pred[~test_df_with_pred['correct']].copy()
errors['true_name'] = errors['label'].map(lambda x: GESTURE_NAMES[x])
errors['pred_name'] = errors['pred'].map(lambda x: GESTURE_NAMES[x])

if len(errors) > 0:
    error_pairs = errors.groupby(['true_name', 'pred_name']).size().sort_values(ascending=False)
    print(f'Total misclassified: {len(errors)} / {len(test_df_with_pred)} ({len(errors)/len(test_df_with_pred)*100:.1f}%)')
    print(f'\nTop misclassification pairs:')
    for (true, pred), count in error_pairs.head(10).items():
        print(f'  {true:15s} -> {pred:15s}  ({count} times)')

In [ ]:
# 可视化：误分类最多的手势对的特征对比
if len(errors) > 0 and len(error_pairs) > 0:
    top_true, top_pred = error_pairs.index[0]
    
    all_feat = pd.read_csv(FEATURES_PATH)
    feat_true = all_feat[all_feat['gesture_name'] == top_true][FEATURE_NAMES]
    feat_pred = all_feat[all_feat['gesture_name'] == top_pred][FEATURE_NAMES]
    
    fig, axes = plt.subplots(3, 3, figsize=(14, 10))
    
    for idx, (ax, feat) in enumerate(zip(axes.flat, FEATURE_NAMES)):
        data_to_plot = [feat_true[feat].values, feat_pred[feat].values]
        bp = ax.boxplot(data_to_plot, labels=[top_true, top_pred], patch_artist=True,
                        widths=0.5)
        bp['boxes'][0].set_facecolor('#64B5F6')
        bp['boxes'][1].set_facecolor('#FF8A65')
        ax.set_title(feat, fontsize=9, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        ax.tick_params(labelsize=8)
    
    fig.suptitle(f'Feature Comparison: Most Confused Pair ({top_true} vs {top_pred})', fontsize=12)
    plt.tight_layout()
    plt.show()

### 如何读这张图

9 个子图分别展示最容易混淆的两个手势在 9 个特征维度上的分布对比：
- **蓝色箱体**：真实手势的特征分布
- **橙色箱体**：被错误预测为的手势的特征分布

如果两个手势在某个特征维度上的箱体高度重叠，说明该特征无法有效区分这两类手势，
这就是模型产生混淆的原因。

**改进方向**：如果所有 9 个特征都难以区分某对手势，可能需要引入新的特征维度
（如频域特征、时序统计量等）来增强区分力。

---
## 6. 模型参数敏感性：树数量

随机森林的主要超参数是决策树数量 `n_estimators`。
下面测试不同树数量对测试准确率的影响，验证 200 棵是否足够。

In [ ]:
tree_counts = [10, 25, 50, 100, 150, 200, 300, 500]
train_accs = []
test_accs = []

for n in tree_counts:
    rf = RandomForestClassifier(
        n_estimators=n,
        max_features='sqrt',
        min_samples_split=3,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    train_accs.append(rf.score(X_train, y_train))
    test_accs.append(rf.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tree_counts, train_accs, 'o-', color='#64B5F6', linewidth=2, label='Train Accuracy')
ax.plot(tree_counts, test_accs, 's-', color='#FF8A65', linewidth=2, label='Test Accuracy')
ax.axhline(test_accs[-1], color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Number of Trees (n_estimators)')
ax.set_ylabel('Accuracy')
ax.set_title('Random Forest: Effect of Tree Count on Accuracy')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.4, 1.05)

# 标注最佳测试准确率
best_idx = np.argmax(test_accs)
ax.annotate(f'Best: {test_accs[best_idx]:.3f} (n={tree_counts[best_idx]})',
            xy=(tree_counts[best_idx], test_accs[best_idx]),
            xytext=(tree_counts[best_idx] + 50, test_accs[best_idx] - 0.05),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=9, color='red')

plt.tight_layout()
plt.show()

print('Tree count vs accuracy:')
for n, tr, te in zip(tree_counts, train_accs, test_accs):
    print(f'  n={n:3d}: train={tr:.3f}, test={te:.3f}')

### 如何读这张图

- **蓝线**：训练集准确率，通常接近 1.0（随机森林容易在训练集上完美拟合）
- **橙线**：测试集准确率，反映模型的真实泛化能力

关键观察：
- 测试准确率通常在 50-100 棵树后趋于稳定，继续增加树的数量收益递减
- 训练和测试准确率之间的差距（gap）反映了过拟合程度
- 如果曲线已经平稳，说明当前的树数量已经足够

---
## 7. 总结

### 当前结果

In [ ]:
print('=' * 50)
print('MODEL SUMMARY')
print('=' * 50)
print(f'Model:           Random Forest ({model.n_estimators} trees)')
print(f'Features:        9D [{", ".join(FEATURE_NAMES)}]')
print(f'Train samples:   {len(y_train)}')
print(f'Test samples:    {len(y_test)}')
print(f'Train Accuracy:  {model.score(X_train, y_train):.3f}')
print(f'Test Accuracy:   {overall_acc:.3f}')
print(f'Macro F1:        {overall_f1:.3f}')
print()
print('Per-environment:')
for env, title in zip(envs, env_titles):
    env_df = test_df_with_pred[test_df_with_pred['env'] == env]
    acc = accuracy_score(env_df['label'], env_df['pred'])
    print(f'  {title:20s}: {acc:.3f}')
print()
print('Top features:')
for rank, idx in enumerate(np.argsort(importances)[::-1][:3], 1):
    print(f'  {rank}. {FEATURE_NAMES[idx]:12s} ({importances[idx]:.3f})')

### 分析与展望

**当前性能：**
- 整体测试准确率约 67%，对于 10 分类任务（随机基线 10%）有明显的区分能力
- 训练准确率 100% 但测试准确率较低，存在过拟合现象
- 部分手势（如 go_the_way、3）分类效果很好，另一些（如 4、2）较差

**过拟合原因分析：**
- 样本量有限（训练集约 400 个），9 维特征空间下随机森林容易记住训练样本
- 部分手势的信号特征本身相似，9 维特征可能不足以完全区分

**可能的改进方向：**
- **增加特征维度**：加入频域特征（如主频率、频带能量比）、时域统计量（如峰值因子、过零率）
- **特征选择**：去除重要度低的特征，减少噪声干扰
- **超参数调优**：限制树的深度（max_depth）减轻过拟合
- **数据增强**：对切分出的片段做时间平移、缩放等增强
- **交叉验证**：使用 k-fold CV 更稳定地评估模型性能